In [145]:
import os, io, re, zipfile, requests
import pandas as pd

In [146]:
#french fama download
OUTDIR = "data"
START_DAY = "19930101"  # keep daily data from this date forward (YYYYMMDD)

BASE = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp"

In [147]:
DATASETS = {
    "10_industry_daily": [
        f"{BASE}/10_Industry_Portfolios_Daily_CSV.zip",
        f"{BASE}/10_Industry_Portfolios_Daily.zip",
    ],
    "30_industry_daily": [
        f"{BASE}/30_Industry_Portfolios_Daily_CSV.zip",
        f"{BASE}/30_Industry_Portfolios_Daily.zip",
    ],
}

In [148]:
DATE_RE = re.compile(r"^\s*(\d{8})\s*,")  # YYYYMMDD at first column

In [149]:
def download_zip_any(urls):
    last_err = None
    for u in urls:
        try:
            r = requests.get(u, timeout=60)
            if r.status_code == 200 and r.content.startswith(b"PK"):
                return io.BytesIO(r.content), u
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to download: {urls} (last_err={last_err})")

In [150]:
def read_first_csv_or_txt(zip_bytes):
    with zipfile.ZipFile(zip_bytes) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith((".csv", ".txt"))]
        if not names:
            raise RuntimeError("No CSV/TXT found in zip.")
        with zf.open(names[0]) as f:
            raw = f.read()
            for enc in ("latin-1", "utf-8"):
                try:
                    return raw.decode(enc)
                except UnicodeDecodeError:
                    continue
            return raw.decode("utf-8", errors="ignore")

In [151]:
def extract_first_panel(ff_text):
    """
    Find the first contiguous block of daily rows (YYYYMMDD, ...).
    Return header line and data lines only.
    """
    lines = ff_text.splitlines()
    # locate first data row
    start = None
    for i, ln in enumerate(lines):
        if DATE_RE.match(ln):
            start = i
            break
    if start is None:
        raise RuntimeError("No daily data rows (YYYYMMDD,...) found.")
    # header is the last non-empty line before 'start'
    hdr_idx = start - 1
    while hdr_idx >= 0 and lines[hdr_idx].strip() == "":
        hdr_idx -= 1
    header = lines[hdr_idx] if hdr_idx >= 0 else None

    # collect contiguous data rows
    data_rows = []
    for ln in lines[start:]:
        if DATE_RE.match(ln):
            data_rows.append(ln)
        else:
            break

    # if header looks bad/missing, synthesize from first row width
    if not header or header.count(",") + 1 != data_rows[0].count(",") + 1:
        ncols = len(data_rows[0].split(","))
        header = ",".join(["Date"] + [f"Col{i}" for i in range(2, ncols + 1)])

    return header, data_rows

In [152]:
def to_df(header, data_rows):
    mini = header + "\n" + "\n".join(data_rows) + "\n"
    df = pd.read_csv(io.StringIO(mini))
    # normalize col names
    first = df.columns[0]
    df = df.rename(columns={first: "Date"})
    df.columns = [c.strip() for c in df.columns]
    # keep from 1993-01-01
    df["Date"] = df["Date"].astype(str).str.strip()
    df = df[df["Date"] >= START_DAY].reset_index(drop=True)
    # numeric columns
    for c in df.columns:
        if c != "Date":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

In [153]:
def save_csv(df, key):
    os.makedirs(OUTDIR, exist_ok=True)
    out = os.path.join(OUTDIR, f"ff_{key.replace('_industry_daily','')}.csv")
    df.to_csv(out, index=False)
    return out

In [154]:
def process(key, urls):
    zip_bytes, used = download_zip_any(urls)
    text = read_first_csv_or_txt(zip_bytes)
    header, rows = extract_first_panel(text)
    df = to_df(header, rows)
    path = save_csv(df, key)
    print(f"[ok] {key}: {used} -> {path} (rows={len(df)}, cols={len(df.columns)})")

In [155]:
if __name__ == "__main__":
    for k, urls in DATASETS.items():
        try:
            process(k, urls)
        except Exception as e:
            print(f"[fail] {k}: {e}")

[ok] 10_industry_daily: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/10_Industry_Portfolios_Daily_CSV.zip -> data/ff_10.csv (rows=8222, cols=11)
[ok] 30_industry_daily: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/30_Industry_Portfolios_Daily_CSV.zip -> data/ff_30.csv (rows=8222, cols=31)


In [156]:
# vix download
URL = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv"
RAW = "VIX_History_raw.csv"
OUT = "data/vix.csv"
START = "1993-01-01"

In [157]:
with requests.get(URL, stream=True, timeout=60) as r:
    r.raise_for_status()
    with open(RAW, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 15):
            if chunk:
                f.write(chunk)

print(f"Downloaded -> {RAW} ({os.path.getsize(RAW):,} bytes)")

Downloaded -> VIX_History_raw.csv (461,038 bytes)


In [158]:
df = pd.read_csv(RAW)
df.columns = [c.strip().title() for c in df.columns]  # standardize names

if "Date" not in df.columns:
    df.rename(columns={df.columns[0]: "Date"}, inplace=True)

if "Close" not in df.columns:
    if "Vix Close" in df.columns:
        df.rename(columns={"Vix Close": "Close"}, inplace=True)
    elif df.shape[1] >= 2:
        df.rename(columns={df.columns[1]: "Close"}, inplace=True)
    else:
        raise ValueError("Couldn't find a Close column in the downloaded CSV.")

df = df[["Date", "Close"]].copy()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Close"] = pd.to_numeric(df["Close"], errors="coerce")

In [159]:
clean = (
    df[(df["Date"] >= START) & df["Close"].notna()]
    .sort_values("Date")
    .reset_index(drop=True)
)

clean.to_csv(OUT, index=False)
print(f"Wrote -> {OUT} (rows={len(clean)})")
clean.head()

Wrote -> data/vix.csv (rows=8286)


,Date,Close
0,1993-01-04,13.36
1,1993-01-05,13.35
2,1993-01-06,13.37
3,1993-01-07,14.72
4,1993-01-08,13.77


In [160]:
try:
    os.remove(RAW)
    print(f"Deleted raw -> {RAW}")
except OSError as e:
    print(f"WARNING: could not delete {RAW}: {e}")

Deleted raw -> VIX_History_raw.csv


In [161]:
# spy download
import yfinance as yf
from curl_cffi import requests

RAW = "SPY_raw_yfinance.csv"
OUT = "data/spy.csv"
START = "1993-01-01"

In [162]:
session = requests.Session(impersonate="chrome")
df = yf.download("SPY", start=START, auto_adjust=False, interval="1mo", session=session)

[*********************100%***********************]  1 of 1 completed


In [163]:
df.head()

Price,Adj Close,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY,SPY
Date,,,,,,
1993-01-01,24.313049,43.93750,43.96875,43.75000,43.96875,1003200
1993-02-01,24.572414,44.40625,45.12500,42.81250,43.96875,5417600
1993-03-01,25.004736,45.18750,45.84375,44.21875,44.56250,3019200
1993-04-01,24.479990,44.03125,45.25000,43.28125,45.25000,2697200
1993-05-01,25.140196,45.21875,45.65625,43.84375,44.09375,1808000


In [164]:
df=df.drop(['Close','High','Low','Open','Volume'],axis=1)
df.head()

Price,Adj Close
Ticker,SPY
Date,
1993-01-01,24.313049
1993-02-01,24.572414
1993-03-01,25.004736
1993-04-01,24.479990
1993-05-01,25.140196


In [165]:
df.to_csv(OUT, index=True)

In [166]:
URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors.CSV"
OUTDIR = "data"
OUTCSV = os.path.join(OUTDIR, "tbill_monthly.csv")

In [167]:
def fetch_ff_rf_monthly() -> pd.Series:
    r = requests.get(URL, timeout=60)
    r.raise_for_status()
    # Ken French files are often Latin-1; fall back if utf-8 fails
    try:
        text = r.content.decode("utf-8")
    except UnicodeDecodeError:
        text = r.content.decode("latin-1")
    lines = text.splitlines()

    def parse_block(start_idx: int, end_idx: int) -> pd.Series:
        block = "\n".join(lines[start_idx:end_idx])
        df = pd.read_csv(io.StringIO(block))
        # Expect columns: DATE, Mkt-RF, SMB, HML, RF (percents)
        df["date"] = pd.to_datetime(df["DATE"].astype(str), format="%Y%m", errors="coerce")
        df = df.dropna(subset=["date"])
        rf = (df.set_index("date")["RF"] / 100.0).rename("R_tbill")
        rf.index = rf.index.to_period("M").to_timestamp("M")
        return rf

    # --- Strategy A: find explicit header and "Annual Factors" footer ---
    try:
        header_idx = next(
            i for i, ln in enumerate(lines)
            if ("DATE" in ln) and ("Mkt-RF" in ln) and ("RF" in ln)
        )
        # monthly table ends right before the "Annual Factors:" marker (if present)
        try:
            footer_idx = next(i for i, ln in enumerate(lines) if ln.strip().startswith("Annual Factors:")) - 1
        except StopIteration:
            # fallback: stop before the first blank line after header
            j = header_idx + 1
            while j < len(lines) and lines[j].strip():
                j += 1
            footer_idx = j
        rf = parse_block(header_idx, footer_idx)
        if not rf.empty:
            return rf
    except StopIteration:
        pass

    # --- Strategy B: detect first numeric YYYYMM line and read contiguous numeric rows ---
    num_re = re.compile(r"^\s*\d{6}([,\s].*)?$")
    try:
        first_num = next(i for i, ln in enumerate(lines) if num_re.match(ln))
    except StopIteration:
        raise RuntimeError("Could not locate the monthly table in the Fama–French file.")

    # find the last contiguous numeric line
    last = first_num
    while last + 1 < len(lines) and num_re.match(lines[last + 1]):
        last += 1

    # synthesize a header (DATE,Mkt-RF,SMB,HML,RF) and parse
    numeric_block = "DATE,Mkt-RF,SMB,HML,RF\n" + "\n".join(lines[first_num:last+1])
    df = pd.read_csv(io.StringIO(numeric_block))
    df["date"] = pd.to_datetime(df["DATE"].astype(str), format="%Y%m", errors="coerce")
    df = df.dropna(subset=["date"])
    rf = (df.set_index("date")["RF"] / 100.0).rename("R_tbill")
    rf.index = rf.index.to_period("M").to_timestamp("M")
    return rf

In [168]:
rf = fetch_ff_rf_monthly()
rf.to_frame().to_csv(OUTCSV, index_label="date")
print(f"Saved {OUTCSV} ({rf.index.min().date()} → {rf.index.max().date()}, {len(rf)} rows)")

Saved data/tbill_monthly.csv (1926-07-31 → 2022-08-31, 1154 rows)
